# Experiment 07 — Cross-product stability

**Paper reference.** §2.1 of v15.

**Goal.** Fit the chosen linear model **per vehicle** within each (scheme, policy), then look at coefficient agreement and the train-on-vehicle-i, test-on-vehicle-j transfer matrix. If margin-normalization is the right inductive bias, off-diagonal entries shouldn't be much worse than diagonals.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

def fit_per_vehicle(scheme, policy):
    fam, mdl = picked(scheme, policy)
    cols = nv.FEATURE_SPECS[fam]['cols']
    sub = turns.query('scheme == @scheme and policy == @policy')
    out = {}
    for v in sorted(sub['vehicle_id'].unique()):
        sv = sub.query('vehicle_id == @v')
        meta, X, y, _ = nv.build_design_matrix(sv, fam)
        (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
        if len(y_n) < 30: continue
        est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_n, y_n)
        out[v] = {'est': est, 'X': X_n, 'y': y_n, 'meta': m_n,
                  'coefs': dict(zip(cols, est.named_steps['m'].coef_))}
    return out, fam, cols

# Coefficient stability for one (scheme, policy)
sch, pol = 'A', 'PVD'
models, fam, cols = fit_per_vehicle(sch, pol)
coef_per_v = pd.DataFrame({v: m['coefs'] for v, m in models.items()}).T
coef_per_v.index.name = 'vehicle'
print(f'{sch}/{pol} on family {fam} -- per-vehicle standardized coefs:')
display(coef_per_v.round(3))
print('coefficient std / |mean| across vehicles (CV):')
display((coef_per_v.std() / coef_per_v.mean().abs()).round(2))

## Train-on-i, test-on-j transfer matrix

In [ ]:
def transfer_matrix(scheme, policy):
    models, fam, cols = fit_per_vehicle(scheme, policy)
    vs = sorted(models.keys())
    mat = np.full((len(vs), len(vs)), np.nan)
    for i, vi in enumerate(vs):
        est = models[vi]['est']
        for j, vj in enumerate(vs):
            yhat = est.predict(models[vj]['X'])
            mat[i, j] = np.mean(np.abs(yhat - models[vj]['y']))
    return pd.DataFrame(mat, index=[f'V{v}' for v in vs], columns=[f'V{v}' for v in vs])

# Display 6 panels (one per policy, scheme=A)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, pol in zip(axes.ravel(), POLICIES):
    M_tr = transfer_matrix('A', pol)
    im = ax.imshow(M_tr.values, cmap='viridis_r', aspect='auto')
    ax.set_xticks(range(len(M_tr.columns))); ax.set_xticklabels(M_tr.columns, fontsize=8)
    ax.set_yticks(range(len(M_tr.index)));   ax.set_yticklabels(M_tr.index, fontsize=8)
    ax.set_title(f'{pol} (scheme A)')
    for i in range(len(M_tr)):
        for j in range(len(M_tr.columns)):
            ax.text(j, i, f'{M_tr.values[i,j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if M_tr.values[i,j] > M_tr.values.mean() else 'black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.suptitle('Train on row vehicle, test on column vehicle (Y-scale MAE)', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp07_transfer_matrix.png'))
plt.show()

**What to look for.** Diagonals are within-vehicle CV-style fits, off-diagonals are pure transfer. If margin-normalization is doing real work, off-diagonals should be only modestly larger than diagonals. Consistent asymmetry (e.g. V5→others always bad) flags a vehicle whose seller cost is so close to the market price that the margin is a very different regime.